In [ ]:
# [셀 1] GPU 확인 + GroundingDINO 설치
!nvidia-smi
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd GroundingDINO
!pip install -q -e .
!pip install -q supervision

In [ ]:
# [셀 2] GroundingDINO 설치 (수정 버전)
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"  # T4 GPU용

!pip install -q torch torchvision
%cd /content/GroundingDINO
!pip install -q -e .

In [ ]:
# [셀 3] 대안 설치
%cd /content
!pip install -q groundingdino-py
!pip install -q supervision

In [ ]:
import os
os.makedirs("/content/weights", exist_ok=True)

!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -O /content/weights/groundingdino_swint_ogc.pth

from groundingdino.util.inference import load_model, load_image, predict, annotate
import torch

model = load_model(
    "/content/GroundingDINO-PromptSensitivity/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py",
    "/content/weights/groundingdino_swint_ogc.pth",
    device="cuda"
)
print("모델 로드 완료!")

In [ ]:
# [셀 5] transformers 버전 다운그레이드
!pip install -q transformers==4.35.2

In [ ]:
# [재시작 후 통합 실행]
!pip install -q groundingdino-py supervision

from groundingdino.util.inference import load_model, load_image, predict, annotate
import torch

model = load_model(
    "/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py",
    "/content/weights/groundingdino_swint_ogc.pth",
    device="cuda"
)
print("모델 로드 완료!")

In [ ]:
# [셀 2] 테스트 inference
import cv2
import matplotlib.pyplot as plt

# 테스트 이미지 다운로드
!wget -q https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/.asset/demo7.jpg -O /content/demo.jpg

image_source, image = load_image("/content/demo.jpg")

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption="horse. clouds. grass. fence.",
    box_threshold=0.35,
    text_threshold=0.25,
    device="cuda"
)

annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title(f"Detections: {len(boxes)}")
plt.axis('off')
plt.show()

for phrase, logit in zip(phrases, logits):
    print(f"  '{phrase}' | confidence: {logit:.4f}")

In [ ]:
# [셀 2 수정] 다른 URL로 테스트
!wget -q "http://images.cocodataset.org/val2017/000000039769.jpg" -O /content/demo.jpg

# 파일 확인
import os
print(f"파일 크기: {os.path.getsize('/content/demo.jpg')} bytes")

image_source, image = load_image("/content/demo.jpg")

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption="cat. couch. remote.",
    box_threshold=0.35,
    text_threshold=0.25,
    device="cuda"
)

annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)

import cv2
import matplotlib.pyplot as plt
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title(f"Detections: {len(boxes)}")
plt.axis('off')
plt.show()

for phrase, logit in zip(phrases, logits):
    print(f"  '{phrase}' | confidence: {logit:.4f}")

In [ ]:
# [셀 3] 실험 이미지 업로드
from google.colab import files
uploaded = files.upload()

In [ ]:
# [셀 4] scene_01.jpg inference
image_source, image = load_image("/content/scene_01.jpg")

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption="car. electric scooter. traffic cone. person. tree.",
    box_threshold=0.30,
    text_threshold=0.25,
    device="cuda"
)

annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 9))
plt.imshow(annotated_rgb)
plt.title(f"scene_01.jpg | Detections: {len(boxes)}")
plt.axis('off')
plt.show()

for phrase, logit in zip(phrases, logits):
    print(f"  '{phrase}' | confidence: {logit:.4f}")

In [ ]:
# [셀 5] 프롬프트 변형 실험 - 추상도 비교
prompts = [
    "vehicle.",           # 추상
    "car.",               # 카테고리
    "white car.",         # 색상 속성
    "parked white car.",  # 상태+색상
]

fig, axes = plt.subplots(1, 4, figsize=(24, 7))

for ax, prompt in zip(axes, prompts):
    image_source, image = load_image("/content/scene_01.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.30, text_threshold=0.25, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDetections: {len(boxes)}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# [셀 6] 나머지 사진 업로드
from google.colab import files
uploaded = files.upload()  # 772~801.jpg 나머지 전부 선택

In [ ]:
# [셀 7] 전체 사진 한눈에 보기
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

files = ['scene_01.jpg','scene_02.jpg','scene_03.jpg','scene_04.jpg','scene_05.jpg','scene_06.jpg','scene_07.jpg','scene_08.jpg','scene_09.jpg']

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for ax, fname in zip(axes, files):
    img = mpimg.imread(f'/content/{fname}')
    ax.imshow(img)
    ax.set_title(fname, fontsize=12)
    ax.axis('off')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# [셀 8] 환각 테스트 - scene_06.jpg (하늘 사진)
image_source, image = load_image("/content/scene_06.jpg")

hallucination_prompts = ["car.", "person.", "cat.", "airplane."]

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, prompt in zip(axes, hallucination_prompts):
    image_source, image = load_image("/content/scene_06.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDetections: {len(boxes)}', fontsize=11)
    ax.axis('off')

plt.suptitle("환각(Hallucination) 테스트 - scene_06.jpg", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# [셀 9] 저조도 테스트 - scene_07.jpg (어두운 계단)
prompts = ["stairs.", "lamp.", "door.", "wall."]

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, prompt in zip(axes, prompts):
    image_source, image = load_image("/content/scene_07.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDetections: {len(boxes)}', fontsize=11)
    ax.axis('off')

plt.suptitle("Low-light Failure Case - scene_07.jpg", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# [셀 10] 한국 특유 객체 - scene_05.jpg (볼라드 거리)
korean_objects = [
    "bollard.",
    "traffic cone.",
    "electric scooter.",
    "crosswalk.",
    "manhole cover.",
]

fig, axes = plt.subplots(1, 5, figsize=(30, 7))
for ax, prompt in zip(axes, korean_objects):
    image_source, image = load_image("/content/scene_05.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDetections: {len(boxes)}', fontsize=10)
    ax.axis('off')

plt.suptitle("Korean-specific Objects - scene_05.jpg", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# [셀 11] 나머지 사진 일괄 기본 inference
remaining = {
    "scene_02.jpg": "car. person. traffic sign. crosswalk.",
    "scene_03.jpg": "car. person. sign board. truck.",
    "scene_04.jpg": "bus. bicycle. car. tree.",
    "scene_08.jpg": "building. fence. tree. window.",
    "scene_09.jpg": "air conditioner. pipe. gate. wall.",
}

for fname, prompt in remaining.items():
    image_source, image = load_image(f"/content/{fname}")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.30, text_threshold=0.25, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(annotated_rgb)
    plt.title(f"{fname} | '{prompt}' | Detections: {len(boxes)}", fontsize=10)
    plt.axis('off')
    plt.show()

    for phrase, logit in zip(phrases, logits):
        print(f"  '{phrase}' | {logit:.4f}")
    print()

In [ ]:
# [셀 12] COCO 샘플 다운로드
import os
os.makedirs("/content/coco_samples", exist_ok=True)

coco_urls = {
    "coco_street.jpg": "http://images.cocodataset.org/val2017/000000397133.jpg",
    "coco_people.jpg": "http://images.cocodataset.org/val2017/000000459153.jpg",
    "coco_vehicle.jpg": "http://images.cocodataset.org/val2017/000000087038.jpg",
}

for fname, url in coco_urls.items():
    !wget -q {url} -O /content/coco_samples/{fname}
    print(f"Downloaded: {fname}")

In [ ]:
# [셀 13] COCO vs 실생활 비교
import numpy as np

compare_pairs = [
    # (COCO 이미지, 실환경 이미지, 프롬프트, 설명)
    ("coco_vehicle.jpg", "scene_01.jpg", "car.", "차량 탐지"),
    ("coco_street.jpg", "scene_02.jpg", "person. traffic sign.", "사람+표지판"),
    ("coco_people.jpg", "scene_04.jpg", "bus. bicycle.", "버스+자전거"),
]

for coco_f, real_f, prompt, desc in compare_pairs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (fpath, label) in zip(axes, [
        (f"/content/coco_samples/{coco_f}", "COCO (공개 데이터셋)"),
        (f"/content/{real_f}", "실생활 (직접 촬영)")
    ]):
        image_source, image = load_image(fpath)
        boxes, logits, phrases = predict(
            model=model, image=image, caption=prompt,
            box_threshold=0.30, text_threshold=0.25, device="cuda"
        )
        annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0

        ax.imshow(annotated_rgb)
        ax.set_title(f"{label}\n탐지: {len(boxes)}개 | 평균 신뢰도: {avg_conf:.3f}", fontsize=11)
        ax.axis('off')

    plt.suptitle(f"[{desc}] Prompt: '{prompt}'", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# [셀 14] 전체 실험 결과 정리 및 저장
import pandas as pd
import numpy as np

results = [
    # 프롬프트 변형 실험 (scene_01.jpg)
    {"실험": "추상도", "사진": "scene_01.jpg", "프롬프트": "vehicle.", "탐지수": 0, "평균신뢰도": 0.000},
    {"실험": "추상도", "사진": "scene_01.jpg", "프롬프트": "car.", "탐지수": 6, "평균신뢰도": 0.550},
    {"실험": "추상도", "사진": "scene_01.jpg", "프롬프트": "white car.", "탐지수": 1, "평균신뢰도": 0.650},
    {"실험": "추상도", "사진": "scene_01.jpg", "프롬프트": "parked white car.", "탐지수": 1, "평균신뢰도": 0.640},
    # 환각 테스트 (scene_06.jpg)
    {"실험": "환각", "사진": "scene_06.jpg", "프롬프트": "car.", "탐지수": 0, "평균신뢰도": 0.000},
    {"실험": "환각", "사진": "scene_06.jpg", "프롬프트": "airplane.", "탐지수": 1, "평균신뢰도": 0.280},
    # 한국 특유 객체 (scene_05.jpg)
    {"실험": "한국특유", "사진": "scene_05.jpg", "프롬프트": "bollard.", "탐지수": 3, "평균신뢰도": 0.320},
    {"실험": "한국특유", "사진": "scene_05.jpg", "프롬프트": "electric scooter.", "탐지수": 1, "평균신뢰도": 0.280},
    {"실험": "한국특유", "사진": "scene_05.jpg", "프롬프트": "crosswalk.", "탐지수": 2, "평균신뢰도": 0.300},
    {"실험": "한국특유", "사진": "scene_05.jpg", "프롬프트": "manhole cover.", "탐지수": 1, "평균신뢰도": 0.270},
    # COCO vs 실생활
    {"실험": "COCO비교", "사진": "COCO", "프롬프트": "person. traffic sign.", "탐지수": 2, "평균신뢰도": 0.763},
    {"실험": "COCO비교", "사진": "scene_02.jpg", "프롬프트": "person. traffic sign.", "탐지수": 21, "평균신뢰도": 0.467},
    {"실험": "COCO비교", "사진": "COCO", "프롬프트": "bus. bicycle.", "탐지수": 0, "평균신뢰도": 0.000},
    {"실험": "COCO비교", "사진": "scene_04.jpg", "프롬프트": "bus. bicycle.", "탐지수": 2, "평균신뢰도": 0.768},
]

df = pd.DataFrame(results)
df.to_csv("/content/experiment_results.csv", index=False)
print("저장 완료!")
display(df)

In [ ]:
# [셀 15] 결과 시각화 차트
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 차트 1: 추상도별 탐지 수
ax1 = axes[0]
prompts = ["vehicle.", "car.", "white car.", "parked\nwhite car."]
counts = [0, 6, 1, 1]
colors = ['#d9534f', '#5cb85c', '#f0ad4e', '#f0ad4e']
ax1.bar(prompts, counts, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title("Abstraction Level vs Detection Count\n(scene_01.jpg)", fontsize=11)
ax1.set_ylabel("Detection Count")
ax1.set_ylim(0, 8)
for i, v in enumerate(counts):
    ax1.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# 차트 2: 환각 테스트
ax2 = axes[1]
halluc_prompts = ["car.\n(없음)", "person.\n(없음)", "cat.\n(없음)", "airplane.\n(없음)"]
halluc_counts = [0, 0, 0, 1]
colors2 = ['#5cb85c', '#5cb85c', '#5cb85c', '#d9534f']
ax2.bar(halluc_prompts, halluc_counts, color=colors2, edgecolor='white', linewidth=1.5)
ax2.set_title("Hallucination Test (scene_06.jpg)\nSky-only Image", fontsize=11)
ax2.set_ylabel("False Positive Count")
ax2.set_ylim(0, 2)
for i, v in enumerate(halluc_counts):
    label = "FP!" if v > 0 else "OK"
    ax2.text(i, v + 0.05, label, ha='center', fontweight='bold',
             color='red' if v > 0 else 'green')

# 차트 3: COCO vs 실생활 신뢰도 비교
ax3 = axes[2]
x = np.arange(2)
coco_conf = [0.763, 0.000]
real_conf = [0.467, 0.768]
width = 0.35
b1 = ax3.bar(x - width/2, coco_conf, width, label='COCO', color='#337ab7', edgecolor='white')
b2 = ax3.bar(x + width/2, real_conf, width, label='Real-world', color='#e67e22', edgecolor='white')
ax3.set_title("COCO vs Real-world\nAvg Confidence", fontsize=11)
ax3.set_ylabel("Average Confidence")
ax3.set_xticks(x)
ax3.set_xticklabels(["person.\ntraffic sign.", "bus.\nbicycle."])
ax3.set_ylim(0, 1.0)
ax3.legend()
for bar in b1:
    h = bar.get_height()
    if h > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.3f}', ha='center', fontsize=9)
for bar in b2:
    h = bar.get_height()
    if h > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.3f}', ha='center', fontsize=9)

plt.suptitle("GroundingDINO Prompt Sensitivity Analysis - Key Results", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/results_chart.png", dpi=150, bbox_inches='tight')
plt.show()
print("차트 저장 완료!")

In [ ]:
# [셀 16] 프롬프트 형식 비교 (scene_04.jpg - 버스+자전거)
format_prompts = [
    "bus. bicycle.",                          # sub-sentence
    "bus and bicycle",                        # 자연어 연결
    "a blue bus on the road",                 # referring - 버스
    "a green bicycle parked on the sidewalk", # referring - 자전거
]

fig, axes = plt.subplots(1, 4, figsize=(28, 7))
for ax, prompt in zip(axes, format_prompts):
    image_source, image = load_image("/content/scene_04.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDetections: {len(boxes)} | Avg: {avg_conf:.3f}', fontsize=9)
    ax.axis('off')

plt.suptitle("Prompt Format Comparison - scene_04.jpg", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# [셀 17] Threshold 민감도 분석
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
det_counts = []
avg_confs = []

for thresh in thresholds:
    image_source, image = load_image("/content/scene_02.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption="car. person. traffic sign.",
        box_threshold=thresh, text_threshold=0.20, device="cuda"
    )
    det_counts.append(len(boxes))
    avg_confs.append(np.mean(logits.numpy()) if len(logits) > 0 else 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(thresholds, det_counts, 'o-', color='steelblue', linewidth=2.5, markersize=8)
ax1.set_xlabel('Box Threshold')
ax1.set_ylabel('Detection Count')
ax1.set_title('Threshold vs Detection Count\n(scene_02.jpg)')
ax1.grid(True, alpha=0.3)
for x, y in zip(thresholds, det_counts):
    ax1.annotate(str(y), (x, y), textcoords="offset points", xytext=(0,8), ha='center')

ax2.plot(thresholds, avg_confs, 's-', color='coral', linewidth=2.5, markersize=8)
ax2.set_xlabel('Box Threshold')
ax2.set_ylabel('Average Confidence')
ax2.set_title('Threshold vs Avg Confidence\n(scene_02.jpg)')
ax2.grid(True, alpha=0.3)
for x, y in zip(thresholds, avg_confs):
    if y > 0:
        ax2.annotate(f'{y:.3f}', (x, y), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8)

plt.suptitle("Threshold Sensitivity Analysis", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/threshold_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 18] Query-text 유사도 분포 (핵심 차별화)
import torch

def get_query_similarity(model, image_path, prompt):
    image_source, image = load_image(image_path)
    model.eval()
    with torch.no_grad():
        caption = prompt.lower().strip()
        if not caption.endswith('.'):
            caption += '.'
        outputs = model(image[None].to("cuda"), captions=[caption])
    logits = outputs["pred_logits"].sigmoid()[0].cpu()
    max_sims = logits.max(dim=1)[0].numpy()
    return max_sims

compare_prompts = ["vehicle.", "car.", "white car.", "bus. bicycle."]
colors = ['#d9534f', '#5cb85c', '#f0ad4e', '#337ab7']

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for ax, prompt, color in zip(axes, compare_prompts, colors):
    sims = get_query_similarity(model, "/content/scene_01.jpg", prompt)
    ax.hist(sims, bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(x=0.35, color='black', linestyle='--', linewidth=1.5, label='threshold=0.35')
    above = (sims > 0.35).sum()
    ax.set_title(f'"{prompt}"\nAbove threshold: {above}', fontsize=10)
    ax.set_xlabel('Max Text Similarity')
    ax.set_ylabel('Query Count')
    ax.legend(fontsize=8)

plt.suptitle("Query-Text Similarity Distribution per Prompt\n(Language-Guided Query Selection 내부 동작)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/query_similarity.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 19] 모든 결과 파일 확인
import os
result_files = [f for f in os.listdir("/content")
                if f.endswith(('.png', '.csv'))]
print("저장된 결과 파일들:")
for f in sorted(result_files):
    size = os.path.getsize(f"/content/{f}") / 1024
    print(f"  {f} ({size:.1f} KB)")

In [ ]:
# [셀 20] 저조도 정량 비교
# 밝은 사진(scene_01.jpg) vs 어두운 사진(scene_07.jpg) 동일 카테고리 신뢰도 비교

compare_data = [
    ("stairs", "scene_01.jpg", "scene_07.jpg"),
    ("wall", "scene_01.jpg", "scene_07.jpg"),
    ("door", "scene_01.jpg", "scene_07.jpg"),
    ("lamp", "scene_01.jpg", "scene_07.jpg"),
]

results_lighting = []

for obj, bright_f, dark_f in compare_data:
    for fname, condition in [(bright_f, "bright"), (dark_f, "dark")]:
        image_source, image = load_image(f"/content/{fname}")
        boxes, logits, phrases = predict(
            model=model, image=image, caption=f"{obj}.",
            box_threshold=0.20, text_threshold=0.15, device="cuda"
        )
        avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
        results_lighting.append({
            "object": obj,
            "condition": condition,
            "file": fname,
            "detections": len(boxes),
            "avg_confidence": round(avg_conf, 4)
        })

df_light = pd.DataFrame(results_lighting)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

objects = ["stairs", "wall", "door", "lamp"]
bright_confs = [df_light[(df_light.object==o)&(df_light.condition=="bright")]["avg_confidence"].values[0] for o in objects]
dark_confs   = [df_light[(df_light.object==o)&(df_light.condition=="dark")]["avg_confidence"].values[0] for o in objects]

x = np.arange(len(objects))
w = 0.35
axes[0].bar(x - w/2, bright_confs, w, label='Bright (scene_01.jpg)', color='#f39c12')
axes[0].bar(x + w/2, dark_confs,   w, label='Dark (scene_07.jpg)',   color='#2c3e50')
axes[0].set_xticks(x)
axes[0].set_xticklabels(objects)
axes[0].set_ylabel("Avg Confidence")
axes[0].set_title("Lighting Condition vs Confidence")
axes[0].legend()
axes[0].set_ylim(0, 0.8)
for i, (b, d) in enumerate(zip(bright_confs, dark_confs)):
    axes[0].text(i - w/2, b + 0.01, f'{b:.3f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, d + 0.01, f'{d:.3f}', ha='center', fontsize=8)

bright_dets = [df_light[(df_light.object==o)&(df_light.condition=="bright")]["detections"].values[0] for o in objects]
dark_dets   = [df_light[(df_light.object==o)&(df_light.condition=="dark")]["detections"].values[0] for o in objects]
axes[1].bar(x - w/2, bright_dets, w, label='Bright (scene_01.jpg)', color='#f39c12')
axes[1].bar(x + w/2, dark_dets,   w, label='Dark (scene_07.jpg)',   color='#2c3e50')
axes[1].set_xticks(x)
axes[1].set_xticklabels(objects)
axes[1].set_ylabel("Detection Count")
axes[1].set_title("Lighting Condition vs Detection Count")
axes[1].legend()

plt.suptitle("Low-light vs Bright: Quantitative Comparison", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/lighting_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

display(df_light)

In [ ]:
# [셀 21] 정량 비교 테이블 (보고서용)
summary_data = [
    # 추상도 실험
    {"Category": "Abstraction", "Prompt": "vehicle.", "Image": "scene_01.jpg", "Detections": 0, "Avg_Conf": 0.000, "Note": "Semantic mismatch"},
    {"Category": "Abstraction", "Prompt": "car.", "Image": "scene_01.jpg", "Detections": 6, "Avg_Conf": 0.550, "Note": "Baseline"},
    {"Category": "Abstraction", "Prompt": "white car.", "Image": "scene_01.jpg", "Detections": 1, "Avg_Conf": 0.650, "Note": "Color filter"},
    {"Category": "Abstraction", "Prompt": "parked white car.", "Image": "scene_01.jpg", "Detections": 1, "Avg_Conf": 0.640, "Note": "State ignored"},
    # 프롬프트 형식
    {"Category": "Format", "Prompt": "bus. bicycle.", "Image": "scene_04.jpg", "Detections": 2, "Avg_Conf": 0.768, "Note": "Sub-sentence"},
    {"Category": "Format", "Prompt": "bus and bicycle", "Image": "scene_04.jpg", "Detections": 4, "Avg_Conf": 0.384, "Note": "Natural lang"},
    {"Category": "Format", "Prompt": "a blue bus on the road", "Image": "scene_04.jpg", "Detections": 1, "Avg_Conf": 0.873, "Note": "Referring"},
    # 환각
    {"Category": "Hallucination", "Prompt": "airplane.", "Image": "scene_06.jpg", "Detections": 1, "Avg_Conf": 0.280, "Note": "False Positive"},
    {"Category": "Hallucination", "Prompt": "car.", "Image": "scene_06.jpg", "Detections": 0, "Avg_Conf": 0.000, "Note": "Correct rejection"},
    # COCO vs 실생활
    {"Category": "Domain Gap", "Prompt": "person. traffic sign.", "Image": "COCO", "Detections": 2, "Avg_Conf": 0.763, "Note": "COCO"},
    {"Category": "Domain Gap", "Prompt": "person. traffic sign.", "Image": "scene_02.jpg", "Detections": 21, "Avg_Conf": 0.467, "Note": "Real-world"},
    # 한국 특유
    {"Category": "Korean-specific", "Prompt": "bollard.", "Image": "scene_05.jpg", "Detections": 3, "Avg_Conf": 0.320, "Note": "Under-detection"},
    {"Category": "Korean-specific", "Prompt": "manhole cover.", "Image": "scene_05.jpg", "Detections": 1, "Avg_Conf": 0.270, "Note": "Novel object"},
    {"Category": "Korean-specific", "Prompt": "crosswalk.", "Image": "scene_05.jpg", "Detections": 2, "Avg_Conf": 0.300, "Note": "Tactile paving FP"},
]

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("/content/full_summary_table.csv", index=False)

# 카테고리별 색상 표시
print("=== 전체 실험 결과 요약 테이블 ===")
display(df_summary)
print("\n저장 완료: full_summary_table.csv")

In [ ]:
# [셀 22] False Positive 박스 위치 분석 (scene_06.jpg airplane)
import matplotlib.patches as patches

image_source, image = load_image("/content/scene_06.jpg")
boxes, logits, phrases = predict(
    model=model, image=image, caption="airplane.",
    box_threshold=0.25, text_threshold=0.20, device="cuda"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 원본 이미지 + bbox
ax1 = axes[0]
ax1.imshow(image_source)
h, w = image_source.shape[:2]
for box, logit, phrase in zip(boxes, logits, phrases):
    cx, cy, bw, bh = box.tolist()
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    bw_px = int(bw * w)
    bh_px = int(bh * h)
    rect = patches.Rectangle((x1, y1), bw_px, bh_px,
                               linewidth=2, edgecolor='red', facecolor='none')
    ax1.add_patch(rect)
    ax1.text(x1, y1-5, f'{phrase} {logit:.3f}',
             color='red', fontsize=11, fontweight='bold',
             bbox=dict(facecolor='white', alpha=0.7))
ax1.set_title('False Positive: "airplane." on sky image', fontsize=12)
ax1.axis('off')

# 확대 crop
ax2 = axes[1]
if len(boxes) > 0:
    box = boxes[0].tolist()
    cx, cy, bw, bh = box
    x1 = max(0, int((cx - bw/2) * w) - 30)
    y1 = max(0, int((cy - bh/2) * h) - 30)
    x2 = min(w, int((cx + bw/2) * w) + 30)
    y2 = min(h, int((cy + bh/2) * h) + 30)
    crop = image_source[y1:y2, x1:x2]
    ax2.imshow(crop)
    ax2.set_title(f'Crop: FP 발생 영역\n(전선/건물 모서리를 airplane으로 오인)', fontsize=11)
    ax2.axis('off')

plt.suptitle("Hallucination Analysis - False Positive Location", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/fp_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"FP 박스 위치: cx={boxes[0][0]:.3f}, cy={boxes[0][1]:.3f}")
print(f"이미지 내 상대 위치: 가로 {boxes[0][0]*100:.1f}%, 세로 {boxes[0][1]*100:.1f}%")

In [ ]:
# [셀 23] 차량 있는 COCO 이미지로 교체 비교
!wget -q "http://images.cocodataset.org/val2017/000000037777.jpg" -O /content/coco_samples/coco_car2.jpg
!wget -q "http://images.cocodataset.org/val2017/000000252219.jpg" -O /content/coco_samples/coco_indoor.jpg

# 차량 비교 재실행
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (fpath, label) in zip(axes, [
    ("/content/coco_samples/coco_car2.jpg", "COCO (공개 데이터셋)"),
    ("/content/scene_01.jpg", "Real-world (직접 촬영)")
]):
    image_source, image = load_image(fpath)
    boxes, logits, phrases = predict(
        model=model, image=image, caption="car.",
        box_threshold=0.30, text_threshold=0.25, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
    ax.imshow(annotated_rgb)
    ax.set_title(f'{label}\nDetections: {len(boxes)} | Avg Conf: {avg_conf:.3f}', fontsize=11)
    ax.axis('off')

plt.suptitle('Domain Gap: COCO vs Real-world | Prompt: "car."', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/domain_gap_car.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 24] 차량 있는 COCO 이미지 정확히 지정
# COCO val2017에서 차량 포함된 이미지 ID 직접 지정
coco_car_urls = [
    "http://images.cocodataset.org/val2017/000000000285.jpg",  # 차량 장면
    "http://images.cocodataset.org/val2017/000000001584.jpg",  # 도로 장면
    "http://images.cocodataset.org/val2017/000000003661.jpg",  # 주차장
]

for i, url in enumerate(coco_car_urls):
    !wget -q {url} -O /content/coco_samples/coco_test_{i}.jpg

# 각각 확인
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, ax in enumerate(axes):
    img = mpimg.imread(f'/content/coco_samples/coco_test_{i}.jpg')
    ax.imshow(img)
    ax.set_title(f'coco_test_{i}.jpg')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# [셀 25] coco_test_1.jpg (버스) vs scene_04.jpg (버스) 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (fpath, label) in zip(axes, [
    ("/content/coco_samples/coco_test_1.jpg", "COCO (London bus)"),
    ("/content/scene_04.jpg", "Real-world (Korean bus)")
]):
    image_source, image = load_image(fpath)
    boxes, logits, phrases = predict(
        model=model, image=image, caption="bus. person.",
        box_threshold=0.30, text_threshold=0.25, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0

    ax.imshow(annotated_rgb)
    ax.set_title(f'{label}\nDetections: {len(boxes)} | Avg Conf: {avg_conf:.3f}', fontsize=11)
    ax.axis('off')

plt.suptitle('Domain Gap: COCO vs Real-world | Prompt: "bus. person."', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/domain_gap_bus.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 26] scene_02.jpg 과탐지 원인 분석
image_source, image = load_image("/content/scene_02.jpg")
boxes, logits, phrases = predict(
    model=model, image=image,
    caption="car. person. traffic sign.",
    box_threshold=0.30, text_threshold=0.25, device="cuda"
)

# 카테고리별 분류
from collections import Counter
phrase_counts = Counter(phrases)
print("카테고리별 탐지 수:")
for k, v in phrase_counts.items():
    confs = [logits[i].item() for i, p in enumerate(phrases) if p == k]
    print(f"  '{k}': {v}개 | 평균 신뢰도: {np.mean(confs):.3f} | 범위: {min(confs):.3f}~{max(confs):.3f}")

# 신뢰도 분포 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 카테고리별 탐지 수 파이차트
ax1.pie(phrase_counts.values(), labels=phrase_counts.keys(),
        autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71'])
ax1.set_title(f'scene_02.jpg Detection Breakdown\n(Total: {len(boxes)})', fontsize=11)

# 신뢰도 분포
colors_map = {'car': '#3498db', 'person': '#e74c3c', 'traffic sign': '#2ecc71'}
for phrase in phrase_counts.keys():
    confs = [logits[i].item() for i, p in enumerate(phrases) if p == phrase]
    ax2.hist(confs, bins=15, alpha=0.7, label=phrase,
             color=colors_map.get(phrase, 'gray'))
ax2.axvline(x=0.30, color='black', linestyle='--', label='threshold=0.30')
ax2.set_xlabel('Confidence')
ax2.set_ylabel('Count')
ax2.set_title('Confidence Distribution by Category\n(scene_02.jpg)', fontsize=11)
ax2.legend()

plt.suptitle("Over-detection Analysis - scene_02.jpg (33 detections)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/overdetection_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 27] 최종 저장 파일 목록 확인
result_files = sorted([f for f in os.listdir("/content")
                       if f.endswith(('.png', '.csv'))])
print("=== 최종 결과 파일 ===")
for f in result_files:
    size = os.path.getsize(f"/content/{f}") / 1024
    print(f"  {f:40s} {size:7.1f} KB")

In [ ]:
# [셀 28] 프롬프트 순서 효과
order_pairs = [
    ("car. person.", "person. car."),
    ("bus. bicycle.", "bicycle. bus."),
    ("traffic sign. car.", "car. traffic sign."),
]

for prompt1, prompt2 in order_pairs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, prompt in zip(axes, [prompt1, prompt2]):
        image_source, image = load_image("/content/scene_02.jpg")
        boxes, logits, phrases = predict(
            model=model, image=image, caption=prompt,
            box_threshold=0.30, text_threshold=0.25, device="cuda"
        )
        annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
        from collections import Counter
        counts = Counter(phrases)
        ax.imshow(annotated_rgb)
        ax.set_title(f'"{prompt}"\nDetections: {len(boxes)} | Avg: {avg_conf:.3f}\n{dict(counts)}', fontsize=9)
        ax.axis('off')

    plt.suptitle("Prompt Order Effect Test", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# [셀 29] 부정 표현 처리 실험
negation_prompts = [
    ("car.", "scene_01.jpg"),                          # 기본
    ("no car.", "scene_01.jpg"),                       # 부정
    ("empty parking space.", "scene_01.jpg"),          # 우회 표현
    ("person not in a car.", "scene_02.jpg"),          # 복합 부정
    ("bus without passengers.", "scene_04.jpg"),       # 속성 부정
]

fig, axes = plt.subplots(1, 5, figsize=(35, 7))

for ax, (prompt, fname) in zip(axes, negation_prompts):
    image_source, image = load_image(f"/content/{fname}")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDet: {len(boxes)} | Avg: {avg_conf:.3f}', fontsize=9)
    ax.axis('off')

plt.suptitle("Negation Handling Test", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/negation_test.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [셀 30] 색상 속성 매트릭스
colors = ["car.", "white car.", "black car.", "silver car.", "red car.", "blue car."]

results_color = []
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
axes = axes.flatten()

for ax, prompt in zip(axes, colors):
    image_source, image = load_image("/content/scene_01.jpg")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.25, text_threshold=0.20, device="cuda"
    )
    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0

    results_color.append({
        "prompt": prompt, "detections": len(boxes), "avg_conf": round(avg_conf, 4)
    })

    ax.imshow(annotated_rgb)
    ax.set_title(f'"{prompt}"\nDet: {len(boxes)} | Avg: {avg_conf:.3f}', fontsize=11)
    ax.axis('off')

plt.suptitle("Color Attribute Matrix - scene_01.jpg", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/color_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

df_color = pd.DataFrame(results_color)
display(df_color)

In [ ]:
# [셀 31] 동일 프롬프트 9장 일관성 분석
all_images = ["scene_01.jpg","scene_02.jpg","scene_03.jpg","scene_04.jpg","scene_05.jpg",
              "scene_06.jpg","scene_07.jpg","scene_08.jpg","scene_09.jpg"]
prompt = "car."

consistency_results = []
for fname in all_images:
    image_source, image = load_image(f"/content/{fname}")
    boxes, logits, phrases = predict(
        model=model, image=image, caption=prompt,
        box_threshold=0.30, text_threshold=0.25, device="cuda"
    )
    avg_conf = np.mean(logits.numpy()) if len(logits) > 0 else 0
    max_conf = max(logits.numpy()) if len(logits) > 0 else 0
    consistency_results.append({
        "image": fname, "detections": len(boxes),
        "avg_conf": round(avg_conf, 4), "max_conf": round(max_conf, 4)
    })

df_consistency = pd.DataFrame(consistency_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df_consistency.image, df_consistency.detections,
            color='steelblue', edgecolor='white')
axes[0].set_title('Detection Count per Image\nPrompt: "car."', fontsize=11)
axes[0].set_xlabel('Image')
axes[0].set_ylabel('Detection Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(df_consistency.detections):
    axes[0].text(i, v+0.1, str(v), ha='center', fontsize=9)

axes[1].plot(df_consistency.image, df_consistency.avg_conf,
             'o-', color='coral', linewidth=2, markersize=8, label='Avg Conf')
axes[1].plot(df_consistency.image, df_consistency.max_conf,
             's--', color='steelblue', linewidth=2, markersize=8, label='Max Conf')
axes[1].set_title('Confidence per Image\nPrompt: "car."', fontsize=11)
axes[1].set_ylabel('Confidence')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cross-image Consistency Analysis | Prompt: "car."', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/consistency_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

display(df_consistency)

In [ ]:
# [셀 32] 최종 파일 목록
result_files = sorted([f for f in os.listdir("/content")
                       if f.endswith(('.png', '.csv'))])
print("=== 최종 결과 파일 전체 목록 ===")
for f in result_files:
    size = os.path.getsize(f"/content/{f}") / 1024
    print(f"  {f:45s} {size:7.1f} KB")

In [ ]:
# [셀 33] 보고서용 핵심 이미지 다운로드
from google.colab import files

key_images = [
    "/content/results_chart.png",
    "/content/query_similarity.png",
    "/content/threshold_analysis.png",
    "/content/color_matrix.png",
    "/content/negation_test.png",
    "/content/domain_gap_bus.png",
]

for f in key_images:
    files.download(f)

In [ ]:
import os

# 실험 결과 파일 위치 찾기
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(('.png', '.csv', '.ipynb')):
            print(os.path.join(root, f))